# No-RAG Baseline

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()  # 自动读取当前工作目录下的 .env
api_key = os.getenv("OPENAI_API_KEY")

In [15]:
# === Imports & Config ===
import os, json, re, math, pathlib, itertools, collections,getpass
import pandas as pd
import numpy as np
from openai import OpenAI
import matplotlib.pyplot as plt
import fitz

CODEBOOK_MD = "/Users/xinby/Desktop/AI44PT_DESKTOP/data/processed/TheCodingTask.md"
PAPER_PDF = "/Users/xinby/Desktop/AI44PT_DESKTOP/data/processed/sample_paper.pdf"
CLS_MODEL = "gpt-5-2025-08-07"  

# Batch assessment settings
MAX_ITEMS = 1  # Number of items to process at a time (OpenAI API limit), None for all at once
SEED = 7          # Random seed
if not os.environ.get('OPENAI_API_KEY'):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter OPENAI_API_KEY: ")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client ok. Using model:", CLS_MODEL)

CODEBOOK_TEXT = pathlib.Path(CODEBOOK_MD).read_text(encoding="utf-8")
print("Loaded Codebook.md chars:", len(CODEBOOK_TEXT))


OpenAI client ok. Using model: gpt-5-2025-08-07
Loaded Codebook.md chars: 43556


In [6]:
def read_pdf(path: str):
    """
    Read a PDF file and extract text from each page.
    Returns a list of dictionaries with 'page' and 'text' keys.
    """
    doc = fitz.open(path)
    pages = []
    for i, pg in enumerate(doc):
        pages.append({'page': i+1, 'text': pg.get_text('text') or ''})
    return pages

def read_markdown(path: str, heading_pattern: str = r'^#{1,6}\s'):
    """
    Read a markdown file and split it into sections based on headings.
    Each section starts with a heading matching the given pattern.
    Returns a list of dictionaries with 'page' (section index) and 'text'.
    """
    heading_re = re.compile(heading_pattern)
    sections = []
    current_lines = []
    section_idx = 1
    with open(path, 'r', encoding='utf-8') as f:
        for raw_line in f:
            line = raw_line.rstrip('\n')
            if heading_re.match(line.strip()):
                if current_lines:
                    sections.append({'page': section_idx, 'text': '\n'.join(current_lines).strip()})
                    section_idx += 1
                current_lines = [line]
            else:
                current_lines.append(line)
        if current_lines:
            sections.append({'page': section_idx, 'text': '\n'.join(current_lines).strip()})
    if not sections:
        text = pathlib.Path(path).read_text(encoding='utf-8')
        sections.append({'page': 1, 'text': text})
    return sections

cb_pages = read_markdown(CODEBOOK_MD)
paper_pages = read_pdf(PAPER_PDF)
print("Codebook sections:", len(cb_pages), "Sample:")
print(*cb_pages[:1], sep="\n")
print("Paper pages:", len(paper_pages), "Sample:")
print(*paper_pages[:1], sep="\n")

Codebook sections: 13 Sample:
{'page': 1, 'text': '# The Coding Task\n\n***This codebook develops a systematic and standardized way to classify qualitative documents into one of Cashore’s four problem types(4PT) framework*** . It will do so by providing guidance to coders on how to assign Type 1, 2, 3 or 4 status to qualitative text including, but not limited to, peer reviewed journal articles, course syllabi, policy statements, policy analyses, media communications and policy recommendations. This classification effort complements, rather than replaces, careful qualitative application\nof the framework required for uncovering important nuances through deliberative analysis and reflection of highly complex text. ***What the quantitative coding effort does well is to generate useful knowledge about whether the overall patterns identified by an one study or set of studies compare to other cases outside of the study. Coding also permits assessments as to whether there are differences in t

In [ ]:
# === JSON Schema for 4PT Analysis ===
FOURPT_ANALYSIS_SCHEMA = {
    "name": "FourPTAnalysis",
    "schema": {
        "type": "object",
        "properties": {
            "Q1_sustainability_fit": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Does the article fit in the universe of sustainability analyses we seek to assess?"
            },
            "Q2_problems_addressed": {
                "type": "string",
                "description": "What problems or set of problems is the article trying to address?",
                "maxLength": 500
            },
            "Q3_on_ground_problem": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Do the analysis, conclusions, and theories derived from, and directed to, understanding and/or managing a clearly specified 'on the ground' problem or class of problems?"
            },
            "Q4_on_ground_arguments": {
                "type": "string",
                "description": "Provide arguments that supports your response to Q3 (Recall Q3: Does the article address a clearly specified on-ground problem?)",
                "maxLength": 800
            },
            "Q5_on_ground_citations": {
                "type": "string",
                "description": "Provide some key text passages from the article that support your Q3 response (Recall Q3: Does the article address a clearly specified on-ground problem?)",
                "maxLength": 800
            },

            "Q6_beyond_on_ground": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Are the analysis, conclusions, and theories generated to apply beyond understanding, and/or managing a clearly specified 'on the ground' problem or class of problems?"
            },
            "Q7_beyond_arguments": {
                "type": "string",
                "description": "Provide arguments that supports your response to Q6 (Recall Q6: Does the article generate analysis, conclusions, and theories to apply beyond understanding/managing a clearly specified on-ground problem?)",
                "maxLength": 800
            },
            "Q8_beyond_on_ground_citations": {
                "type": "string",
                "description": "Provide some key text passages from the article that support your Q6 response (Recall Q6: Does the article generate analysis, conclusions, and theories to apply beyond understanding/managing a clearly specified on-ground problem?)",
                "maxLength": 800
            },
            "Q9_utility_maximization": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Do the analysis, conclusions, and theories treat individuals, organizations and states as largely self-interested, satisfaction driven entities that seek to maximize some kind of 'utility' outcome?"
            },
            "Q10_utility_arguments": {
                "type": "string",
                "description": "Provide arguments that supports your response to Q9 (Recall Q9: Does the article treat entities as self-interested, utility-maximizing agents?)",
                "maxLength": 800
            },
            "Q11_utility_citations": {
                "type": "string",
                "description": "Provide some key text passages from the article that support your Q9 response (Recall Q9: Does the article treat entities as self-interested, utility-maximizing agents?)",
                "maxLength": 800
            },
            "Q12_beyond_self_interest": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Do the analysis incorporate theories and conclusions incorporate an assessment of individuals, organizations and/or states that extends beyond self-interested satisfaction seeking motivations?"
            },
            "Q13_beyond_self_interest_arguments": {
                "type": "string",
                "description": "Provide arguments that supports your response to Q12 (Recall Q12: Does the article extend beyond self-interested satisfaction seeking motivations?)",
                "maxLength": 800
            },
            "Q14_beyond_self_interest_citations": {
                "type": "string",
                "description": "Provide some key text passages from the article that support your Q12 response (Recall Q12: Does the article extend beyond self-interested satisfaction seeking motivations?)",
                "maxLength": 800
            },
            "Q15_fourpt_type": {
                "type": "string",
                "enum": ["T1", "T2", "T3", "T4"],
                "description": "Your final 4PT Type classification"
            },
            "Q16_evaluation": {
                "type": "string",
                "description": "Difficulty level (1-5) justification for your 4PT type (≤120 characters)",
                "enum":["easy","medium","hard"]
            }
        },
        "required": [
            "Q1_sustainability_fit",
            "Q2_problems_addressed",
            "Q3_on_ground_problem",
            "Q4_on_ground_arguments",
            "Q5_on_ground_citations",
            "Q6_beyond_on_ground",
            "Q7_beyond_arguments",
            "Q8_beyond_on_ground_citations",
            "Q9_utility_maximization",
            "Q10_utility_arguments",
            "Q11_utility_citations",
            "Q12_beyond_self_interest",
            "Q13_beyond_self_interest_arguments",
            "Q14_beyond_self_interest_citations",
            "Q15_fourpt_type",
            "Q16_evaluation"
        ],
        "additionalProperties": False
    },
    "strict": True
}

In [20]:
def analyze_article_fourpt(article_pages: list, codebook_pages: list, model: str = CLS_MODEL):
    """
    使用4PT框架分析文章一次性回答所有问题
    """
    # 将页面内容合并为文本
    article_text = "\n\n".join([f"Page {p['page']}:\n{p['text']}" for p in article_pages])
    codebook_text = "\n\n".join([f"Section {p['page']}:\n{p['text']}" for p in codebook_pages])
    
    prompt = f"""
You are an expert public policy analyst reviewing sustainability research articles. 

**Instructions:**
- Answer ALL questions based on the provided Codebook and Article
- Provide specific citations when requested
- Keep justifications concise and evidence-based
- For Yes/No questions, choose definitively based on evidence

**4PT Codebook:**
{codebook_text}

**Article to Analyze:**
{article_text}

Please analyze this article systematically through the 4PT framework, answering each question thoroughly.
    """.strip()
    
    try:
        print(f"🔍 API Call Info:")
        print(f"  Model: {model}")
        print(f"  Prompt length: {len(prompt)} characters")
        print(f"  Article pages: {len(article_pages)}")
        print(f"  Codebook pages: {len(codebook_pages)}")
        
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_schema", "json_schema": FOURPT_ANALYSIS_SCHEMA},
            max_completion_tokens=2000,
        )
        
        print(f"✅ API Response received")
        print(f"  Response type: {type(resp)}")
        print(f"  Has choices: {hasattr(resp, 'choices') and len(resp.choices) > 0}")
        
        if resp.choices and len(resp.choices) > 0:
            content = resp.choices[0].message.content
            print(f"  Content length: {len(content) if content else 0}")
            print(f"  Content preview: {content[:200] if content else 'None'}...")
            
            if not content:
                print("❌ Error: Response content is empty")
                return None
                
            if content.strip() == "":
                print("❌ Error: Response content is whitespace only")
                return None
            
            try:
                result = json.loads(content)
                print(f"✅ JSON parsing successful")
            except json.JSONDecodeError as json_err:
                print(f"❌ JSON parsing failed: {json_err}")
                print(f"Raw content:\n{content}")
                return None
        else:
            print("❌ Error: No choices in response")
            return None
        
        # 添加一些派生字段便于分析
        result["analysis_metadata"] = {
            "model": model,
            "timestamp": pd.Timestamp.now().isoformat()
        }
        
        return result
        
    except Exception as e:
        print(f"❌ API Call failed: {type(e).__name__}: {e}")
        if hasattr(e, 'response'):
            print(f"Error response: {e.response}")
        return None

In [ ]:
def analyze_article_fourpt(article_pages: list, codebook_pages: list, model: str = CLS_MODEL):
    """
    使用4PT框架分析文章一次性回答所有问题
    """
    # 将页面内容合并为文本
    article_text = "\n\n".join([f"Page {p['page']}:\n{p['text']}" for p in article_pages])
    codebook_text = "\n\n".join([f"Section {p['page']}:\n{p['text']}" for p in codebook_pages])
    
    prompt = f"""
You are an expert public policy analyst reviewing sustainability research articles. 

**Instructions:**
- Answer ALL questions based on the provided Codebook and Article
- Provide specific citations when requested
- Keep justifications concise and evidence-based
- For Yes/No questions, choose definitively based on evidence

**4PT Codebook:**
{codebook_text}

**Article to Analyze:**
{article_text}

Please analyze this article systematically through the 4PT framework, answering each question thoroughly.
    """.strip()
    
    try:
       

        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_schema", "json_schema": FOURPT_ANALYSIS_SCHEMA},
            max_completion_tokens=2000,
        )
        
        result = json.loads(resp.choices[0].message.content)
        
        # 添加一些派生字段便于分析
        result["analysis_metadata"] = {
            "model": model,
            "timestamp": pd.Timestamp.now().isoformat()
        }
        
        return result
        
    except Exception as e:
        print(f"Error in analysis: {e}")
        return None

In [22]:
result = analyze_article_fourpt(paper_pages, cb_pages)

🔍 API Call Info:
  Model: gpt-5-2025-08-07
  Prompt length: 132891 characters
  Article pages: 19
  Codebook pages: 13
✅ API Response received
  Response type: <class 'openai.types.chat.chat_completion.ChatCompletion'>
  Has choices: True
  Content length: 0
  Content preview: None...
❌ Error: Response content is empty


In [14]:
print(*result.items(), sep="\n")

AttributeError: 'NoneType' object has no attribute 'items'

For reference, the human coder's answer for this paper is:
1. Yes
2. Interaction of program design conditions in affecting performance of voluntary programs
3. No
4. Geographical focus in several countries with no clearly specified on the ground problem
5. Looks at the topic of low carbon building and city development across cases in Australia, the Netherlands and the US
6. Yes
7. Explores implications on voluntary program performance of the diffusion network
8. This article has introduced the diffusion network perspective as a critical condition to understand voluntary program performance
9. Analysis on generalizability is valid. Proceed to the next questions on utility.
10. Yes
11. Key objective: determinants of voluntary program performance, which originates from high cost of direct regulatory intervention
12. Voluntary programs aim to change the behavior of individuals and organizations, but without the force of law. They have rapidly become popular governance instruments in situations in which it is too costly or difficult to implement direct regulatory interventions, for instance because of political unwillingness (Darnall & Carmin 2005). They also provide an opportunity to showcase and market desired “beyond compliance” behavior, or to reward leading firms (Saurwein 2011).
13. No
14. Looks at how diffusion network perspective can relate to better voluntary program results
15. The diffusion network perspective, as conceptualized in this article, brings together these insights and argues that the stronger the diffusion network, the more likely it is that a voluntary program will achieve the desired results.
16. Analysis on generalizability is valid. Proceed to the next questions on utility.
17. Type 2
18. 4 - Hard